# Supporting validation/context notebook

This notebook provides supporting validation and contextual evidence for VDR-related analyses (for example, external overlap checks) and is **not** a canonical manuscript-regeneration step.


In [ ]:
import pandas as pd
import pyranges as pr
import re
from pathlib import Path

core_genes = [
    "ADGRE5", "AGR2", "BAG3", "C2CD2", "CXCL2", "DDIT4",
    "DNAJA3", "GTF2B", "HMGCS1", "IARS", "KIF20A",
    "MTHFD2", "NFKBIA", "NPC1", "PCNA", "PHGDH",
    "RAD51C", "SPP1", "TSKU"
]

gtf = pr.read_gtf("gencode.v44.basic.annotation.gtf")

genes = gtf[gtf.Feature == "gene"]

genes_df = genes.df.copy()

genes_df["TSS"] = genes_df.apply(
    lambda row: row["Start"] if row["Strand"] == "+" else row["End"],
    axis=1
)

def clean_gene_symbols(gene_list):
    return sorted({
        g for g in gene_list
        if isinstance(g, str)
        and not g.startswith("ENSG")
        and re.match(r"^[A-Z0-9\-]+$", g)
    })


def get_vdr_genes_from_bed(
    bed_path,
    genes_df,
    window=10_000
):
    # Leer BED
    bed = pd.read_csv(bed_path, sep="\t", header=None, comment="#")
    bed = bed.iloc[:, :3]
    bed.columns = ["Chromosome", "Start", "End"]

    peaks_pr = pr.PyRanges(bed)

    # Ventana alrededor del TSS
    tss_window = genes_df[["Chromosome", "TSS", "gene_name"]].copy()
    tss_window["Start"] = (tss_window["TSS"] - window).clip(lower=0)
    tss_window["End"] = tss_window["TSS"] + window
    tss_window = tss_window[["Chromosome", "Start", "End", "gene_name"]]

    tss_pr = pr.PyRanges(tss_window)

    # Overlap peaks-TSS windows
    overlap = peaks_pr.join(tss_pr)

    vdr_genes = (
        overlap.df["gene_name"]
        .dropna()
        .drop_duplicates()
        .tolist()
    )

    return clean_gene_symbols(vdr_genes)


def overlap_core_with_vdr(
    bed_path,
    genes_df,
    core_genes,
    windows=(10_000, 50_000, 100_000)
):
    results = []

    for window in windows:
        vdr_genes = get_vdr_genes_from_bed(
            bed_path=bed_path,
            genes_df=genes_df,
            window=window
        )

        overlap_genes = sorted(set(core_genes) & set(vdr_genes))

        results.append({
            "bed_file": Path(bed_path).name,
            "window_bp": window,
            "n_vdr_genes": len(vdr_genes),
            "n_core_genes": len(core_genes),
            "n_overlap": len(overlap_genes),
            "overlap_genes": ", ".join(overlap_genes)
        })

    return pd.DataFrame(results)

14. Characterisation of VDR signaling in prostate cancer health disparities (ChIP-Seq) 

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE221826

In [ ]:
bed_path_one = "GSE221826_clusters.RC43N.IgG.VDR.D3.bed"

result_one = overlap_core_with_vdr(
    bed_path=bed_path_one,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_one

bed_path_two = "GSE221826_clusters.RC43t.IgG.VDR.D3.bed"

result_two = overlap_core_with_vdr(
    bed_path=bed_path_two,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_two

In [ ]:
bed_path_three = "GSE221826_clusters.LNCaP.IgG.VDR.D3.bed"

result_three = overlap_core_with_vdr(
    bed_path=bed_path_three,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_three

23. Genome wide VDR binding sites in RWPE1 human prostate epithelial cells
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116843

In [ ]:
bed_path_one = pd.read_excel("GSE116843_RWPE_Peaks_for_GEO_7_2_18.xlsx")

In [ ]:
rwpe = pd.read_excel("GSE116843_RWPE_Peaks_for_GEO_7_2_18.xlsx")

rwpe_vitd = rwpe[rwpe["Peak Height Vitamin D"] > 0].copy()

rwpe_vdr_genes = (
    rwpe_vitd["Gene Symbol"]
    .dropna()
    .astype(str)
    .str.upper()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

overlap_rwpe = sorted(set(core_genes) & set(rwpe_vdr_genes))

len(rwpe_vdr_genes), overlap_rwpe

34. Antagonistic interaction between androgen signaling and vitamin D signaling in prostate cancer [ChIP-Seq]
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE64656

In [ ]:
bed_path_one = "enrichment\GSM1576449_VDR_binding_sites.bed"

result_three = overlap_core_with_vdr(
    bed_path=bed_path_three,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_one

10. Vitamin D receptor (VDR) ChIP-Seq on human colonic organoids stimulated with 1,25(OH)2D3
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE206176

In [ ]:
bed_path_one = "GSE206176_macs2_bdgdiff_VA_vs_EA_c3.0_cond1.bed"

result_three = overlap_core_with_vdr(
    bed_path=bed_path_three,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_one

In [ ]:
bed_path_two = "GSE206176_macs2_bdgdiff_VB_vs_EB_c3.0_cond1.bed"

result_two = overlap_core_with_vdr(
    bed_path=bed_path_two,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_two

21.VDR ChIP-Seq on colonic human organoids
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE107283

In [ ]:
bed_path_one = pd.read_csv("GSM2863694_5diff_c1_vs_c2_c3.0_cond2.bed", sep="\t")

In [ ]:
bed = pd.read_csv(
    "GSM2863694_5diff_c1_vs_c2_c3.0_cond2.bed",
    sep="\t",
    header=None,
    skiprows=1
)

In [ ]:
bed.head()

In [ ]:
bed_path_one = "GSM2863694_5diff_c1_vs_c2_c3.0_cond2.bed"

result_one = overlap_core_with_vdr(
    bed_path=bed_path_one,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_one

In [ ]:
bed_path_two = "GSM2863699_8diff_c1_vs_c2_c3.0_cond2.bed"

result_two = overlap_core_with_vdr(
    bed_path=bed_path_two,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_two

In [ ]:
bed_path_three = "GSM2863694_6diff_c1_vs_c2_c3.0_cond2.bed"

result_three = overlap_core_with_vdr(
    bed_path=bed_path_three,
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

result_three

In [ ]:
bed = pd.read_csv(
    "remap2022_VDR_nr_macs2_hg38_v1_0.bed",
    sep="\t",
    header=None
)

bed = bed[[0,1,2]]
bed.columns = ["chrom", "start", "end"]

In [ ]:
bed.head()

In [ ]:
result_remap = overlap_core_with_vdr(
    bed_path="remap2022_VDR_nr_macs2_hg38_v1_0.bed",
    genes_df=genes_df,
    core_genes=core_genes,
    windows=(10_000, 50_000, 100_000)
)

print(result_remap)